# Coding Exercise 1 (Student Version): Simulating SDEs

In this exercise, you will implement key components for simulating **stochastic differential equations (SDEs)**, and in particular the **Langevin diffusion**.

**How to use this notebook**
- Read each **🧩 Question** in the markdown.
- Fill in the corresponding code cell(s). When you see `raise NotImplementedError("Fill me in!")`, replace it with your implementation.
- Run cells top-to-bottom. Many later sections depend on earlier ones.



In [1]:
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch.func import vmap, jacrev
from abc import ABC, abstractmethod
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# set random seeds for reproducibility
np.random.seed(12345)
torch.manual_seed(12345)

## Section 1: Specify a distribution

We will represent distributions by specifying

- whether they have an explicit (log-)density, and
- whether we can sample from them.

### 🧩 Question 1.1 — Gaussian distribution
Complete the methods in the `Gaussian` class:
- `log_prob(x)` should return the log-density with shape `(batch_size, 1)`
- `sample(num_samples)` should return samples with shape `(num_samples, dim)`

### 🧩 Question 1.2 — Mixture of Gaussians
Complete the methods in the `MixtureOfGaussians` class:
- `log_prob(x)` should implement a numerically stable mixture log-density (hint: `torch.logsumexp`)
- `sample(num_samples)` should sample from the mixture according to the weights

After implementing, you'll visualize densities and samples in the cells below.


In [3]:
class Density(ABC):
    @abstractmethod
    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns the log density at x
        """
        pass
    
    def score(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns the score function at x, i.e. the gradient of the log density
        The advantage of torch is that we can compute the score function using automatic differentiation, without having to derive it analytically
        """
        x = x.unsqueeze(1)  # (batch_size, 1, ...)
        score = vmap(jacrev(self.log_prob))(x)  # (batch_size, 1, 1, 1, ...)
        return score.squeeze((1, 2, 3))  # (batch_size, ...)
    

class Sampleable(ABC):
    @abstractmethod
    def sample(self, num_samples: int):
        """
        Return num_samples samples from the distribution
        """
        pass

In [4]:
# examples of distributions (YOU will fill in key methods)

class Gaussian(torch.nn.Module, Density, Sampleable):
    """
    Multivariate Gaussian with mean and covariance.
    """
    def __init__(self, mean: torch.Tensor, cov: torch.Tensor):
        super().__init__()
        self.mean = mean
        self.cov = cov
        self.inv_cov = torch.linalg.inv(cov)
        self.log_det_cov = torch.logdet(cov)
        self.dim = mean.shape[0]

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q1.1): Return log p(x) with shape (batch_size, 1).

        Hints:
        - You can use torch.distributions.MultivariateNormal
        - Or implement the quadratic form by hand using self.inv_cov and self.log_det_cov
        """
        raise NotImplementedError("Fill me in!")

    def sample(self, num_samples: int) -> torch.Tensor:
        """
        TODO (Q1.1): Draw num_samples samples with shape (num_samples, dim).
        """
        raise NotImplementedError("Fill me in!")


class MixtureOfGaussians(torch.nn.Module, Density, Sampleable):
    """
    Mixture of Gaussians with given means, covariances, and weights.
    """
    def __init__(self, means: torch.Tensor, covs: torch.Tensor, weights: torch.Tensor):
        super().__init__()
        self.means = means
        self.covs = covs
        self.weights = weights / weights.sum()  # normalize weights
        self.num_components = means.shape[0]
        self.dim = means.shape[1]
        self.components = [Gaussian(means[i], covs[i]) for i in range(self.num_components)]

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q1.2): Compute log p(x) for the mixture with shape (batch_size, 1).

        Hint: use a numerically stable computation via torch.logsumexp over components.
        """
        raise NotImplementedError("Fill me in!")

    def sample(self, num_samples: int) -> torch.Tensor:
        """
        TODO (Q1.2): Sample from the mixture according to the weights.
        Return shape: (num_samples, dim).
        """
        raise NotImplementedError("Fill me in!")

    @classmethod
    def random_2D(
        cls, num_components: int, mean_range: float = 10.0, cov_range: float = 5.0
    ) -> "MixtureOfGaussians":
        """Helper (provided): random 2D mixture."""
        means = torch.rand(num_components, 2) * mean_range - mean_range / 2
        covs = torch.stack([torch.diag(torch.rand(2) * cov_range + 0.5) for _ in range(num_components)], dim=0)
        weights = torch.rand(num_components)
        return cls(means, covs, weights)

    @classmethod
    def circle_2D(
        cls, num_components: int, radius: float = 5.0, cov_range: float = 1.0
    ) -> "MixtureOfGaussians":
        """Helper (provided): place mixture means on a circle."""
        angles = torch.linspace(0, 2 * np.pi, num_components + 1)[:-1]  # exclude last (=first)
        means = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * radius
        covs = torch.stack([torch.diag(torch.rand(2) * cov_range + 0.5) for _ in range(num_components)], dim=0)
        weights = torch.rand(num_components)
        return cls(means, covs, weights)


In [5]:
# TODO (Q1.3): Create a dict called `densities` with (at least) three entries:
#   - a Gaussian in 2D
#   - a random mixture of Gaussians in 2D
#   - a symmetric mixture of Gaussians arranged in a circle
#
# Make sure each density is moved to the correct device with `.to(device)`.

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

In [6]:
# same but contour plot
fig, axies = plt.subplots(1, len(densities), figsize=(15, 5))
scale = 10
bins = 100

for ax, (name, density) in zip(axies, densities.items()):
    x = torch.linspace(-scale, scale, bins)
    y = torch.linspace(-scale, scale, bins)
    X, Y = torch.meshgrid(x, y)
    grid = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
    with torch.no_grad():
        log_probs = density.log_prob(grid).cpu().numpy().reshape(bins, bins)
    ax.contourf(X.cpu(), Y.cpu(), log_probs, levels=50, cmap="rainbow")
    ax.set_title(name)
plt.tight_layout()
plt.show()

NameError: name 'densities' is not defined

In [7]:
# same but scatter plot of samples
fig, axies = plt.subplots(1, len(densities), figsize=(15, 5))
num_samples = 1000
for ax, (name, density) in zip(axies, densities.items()):
    samples = density.sample(num_samples).cpu().numpy()
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

NameError: name 'densities' is not defined

## Section 2: Numerical methods to simulate SDEs

Given $T>0$, consider an SDE of the form
$$
d X_t = a(X_t, t)\, dt + \sigma(X_t, t)\, d W_t,
$$
where
$$
a: \mathbb{R}^n \times [0, T] \to \mathbb{R}^n,\qquad
\sigma: \mathbb{R}^n \times [0, T] \to \mathbb{R}^n,
$$
and $W_t$ is standard Brownian motion.

For simplicity, in most scenarios we consider, $\sigma(X_t, t)$ won't depend on $X_t$, so we may write $\sigma_t$.

### 🧩 Question 2.1 — Brownian motion as an SDE
Implement the SDE class for (scaled) Brownian motion:
$$
dX_t = \sigma \, dW_t, \qquad X_0 = 0.
$$
Complete `BrownianMotion.drift` and `BrownianMotion.diffusion`.


In [8]:
# class to encapsulate SDE definition
class SDE(ABC):
    @abstractmethod
    def drift(self, x, t):
        """
        Compute the drift term a(x, t) of the SDE at state x and time t
        """
        pass

    @abstractmethod
    def diffusion(self, x, t):
        """
        Compute the diffusion term sigma(x, t) of the SDE at state x and time t
        """
        pass


#### Define Brownian motion

Recall the standard Brownian motion:
$$
dX_t = dW_t, \qquad X_0 = 0.
$$

A scaled version is:
$$
dX_t = \sigma \, dW_t.
$$

You will implement it as an `SDE` below.


In [9]:
class BrownianMotion(SDE):
    def __init__(self, sigma: float = 1.0):
        self.sigma = sigma

    def drift(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q2.1): Implement the drift a(x,t) for Brownian motion.
        """
        raise NotImplementedError("Fill me in!")

    def diffusion(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q2.1): Implement the diffusion term sigma(x,t).
        For (scaled) Brownian motion, it should be constant in x and t.
        """
        raise NotImplementedError("Fill me in!")


## Section 3: Euler–Maruyama discretization

To simulate an SDE, discretize time. With step size $h=\Delta t$, Euler–Maruyama gives:
$$
X_{t+h} = X_t + h\, a(X_t,t) + \sqrt{h}\, \sigma(X_t,t)\, Z_t,
$$
where $Z_t \sim \mathcal{N}(0, I_n)$.

### 🧩 Question 3.0 — Simulator loop
Complete `Simulator.simulate` and `Simulator.simulate_trajectory`. These should repeatedly call `self.step(...)` over the time grid.

### 🧩 Question 3.1 — Implement Euler–Maruyama
Complete `EulerMaruyama.step` so that it performs **one** Euler–Maruyama update.

Then, in the next section, you will use it to simulate Brownian motion and the OU process.


In [10]:
class Simulator(ABC):
    @abstractmethod
    def step(self, x, t, dt):
        """
        Simulate one step of the SDE from state x at time t with step size dt
        """
        pass

    @torch.no_grad()
    def simulate(self, x: torch.Tensor, ts: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q3.0): Simulate the SDE starting from initial state x over the time points in ts.
        Return only the final state x_T.

        Hint: loop over i = 0,...,len(ts)-2 and use dt = ts[i+1] - ts[i].
        """
        raise NotImplementedError("Fill me in!")

    @torch.no_grad()
    def simulate_trajectory(self, x: torch.Tensor, ts: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q3.0): Simulate the SDE and return the full trajectory with shape:
            (batch_size, len(ts), dim)

        Hint: store x at each time step in a Python list, then stack.
        """
        raise NotImplementedError("Fill me in!")


In [11]:
class EulerMaruyama(Simulator):
    def __init__(self, sde: SDE):
        self.sde = sde

    def step(self, x: torch.Tensor, t: torch.Tensor, dt: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q3.1): Perform one Euler–Maruyama update:
            x_{t+dt} = x_t + a(x_t,t) dt + sigma(x_t,t) * sqrt(dt) * z
        where z ~ N(0, I).

        Make sure the returned tensor has the same shape as x.
        """
        raise NotImplementedError("Fill me in!")


#### Simulate Brownian motion

### 🧩 Question 3.2 — Simulate trajectories and check variance
Using your Euler–Maruyama implementation:

1. Simulate multiple trajectories of 1D Brownian motion.
2. Plot the trajectories.
3. Compute the empirical variance at the final time and compare it to the theoretical value
   $$\operatorname{Var}(X_T) = \sigma^2 T.$$


In [12]:
# TODO (Q3.2): Simulate and plot trajectories of 1D Brownian motion.
#
# Suggested steps:
# 1) choose sigma, number of trajectories, time horizon T, and number of time steps
# 2) instantiate BrownianMotion and EulerMaruyama
# 3) define x0 with shape (num_trajs, 1) (all zeros)
# 4) create ts = torch.linspace(0, T, num_steps+1)
# 5) trajectories = simulator.simulate_trajectory(x0, ts)
# 6) plot trajectories

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

In [13]:
# TODO (Q3.2): Compare empirical variance vs. theoretical variance for Brownian motion.
#
# Theoretical: Var(X_T) = sigma^2 * T

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

#### Simulate Ornstein–Uhlenbeck (OU) process

Recall the OU process:
$$
dX_t = -\theta X_t\, dt + \sigma \, dW_t, \qquad X_0 = x_0.
$$

### 🧩 Question 3.3 — Implement and simulate OU
1. Implement the OU SDE in `OrnsteinUhlenbeck`.
2. Simulate trajectories using Euler–Maruyama.
3. Observe how the variance behaves over time (compare to Brownian motion).


In [14]:
# define OU process
class OrnsteinUhlenbeck(SDE):
    def __init__(self, theta: float = 1.0, sigma: float = 1.0):
        self.theta = theta
        self.sigma = sigma

    def drift(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q3.3): Implement drift for OU: a(x,t) = -theta * x
        """
        raise NotImplementedError("Fill me in!")

    def diffusion(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q3.3): Implement diffusion for OU (constant in x,t).
        """
        raise NotImplementedError("Fill me in!")


In [15]:
# TODO (Q3.3): Simulate and plot trajectories of the 1D OU process.
#
# Suggested steps:
# 1) choose theta and sigma (try sigma = sqrt(2*theta) to match the stationary variance in Section 4)
# 2) instantiate OrnsteinUhlenbeck and EulerMaruyama
# 3) pick an initial distribution for x0 (e.g. uniform on [0,1])
# 4) create a time grid ts
# 5) simulate trajectories and plot them

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

In [16]:
# TODO (Q3.3): Compute the empirical variance at the final time for the OU process.
#
# Optional: try increasing the time horizon to see whether the variance stabilizes.

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

## Section 4: Simulate Langevin dynamics

For a target density $p$, the Langevin SDE is:
$$
dX_t = \tfrac12 \sigma^2 \, \nabla \log p(X_t)\, dt + \sigma \, dW_t, \qquad X_0=x_0.
$$

### 🧩 Question 4.1 — Implement the Langevin SDE
Complete the `Langevin` class below:
- Implement the drift using the score function $\nabla \log p(x)$.
- Implement the diffusion term.

### 🧩 Question 4.2 — OU as a special Langevin case (written)
Show that the OU process is a special kind of Langevin process with the choice:
$$
p(x) = \mathcal{N}\Big(0, \tfrac{\sigma^2}{2\theta}\Big).
$$

Afterwards, you will simulate Langevin dynamics for multimodal targets.


In [17]:
# define Langevin SDE
class Langevin(SDE):
    def __init__(self, tar_density: Density, sigma: float = 1.0):
        self.tar_density = tar_density
        self.sigma = sigma

    def drift(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q4.1): Implement Langevin drift:
            a(x,t) = 0.5 * sigma^2 * score(x)
        where score(x) = ∇ log p(x).
        """
        raise NotImplementedError("Fill me in!")

    def diffusion(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        TODO (Q4.1): Implement the diffusion term (typically constant).
        """
        raise NotImplementedError("Fill me in!")


In [18]:
# TODO (Q4.1): Simulate Langevin dynamics for a multimodal target density.
#
# Suggested steps:
# 1) choose a target density from `densities` (e.g., random or symmetric mixture)
# 2) instantiate Langevin(tar_density=..., sigma=...)
# 3) simulate with EulerMaruyama for a sufficiently long time
# 4) visualize how particles move over time (e.g., scatter every k steps)

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

In [19]:
# Helper (provided): plot final distribution of samples vs. true density
def plot_final_dist(trajectories, tar_density):
    x = torch.linspace(-10, 10, 100)
    y = torch.linspace(-10, 10, 100)
    X, Y = torch.meshgrid(x, y)
    grid = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
    with torch.no_grad():
        log_probs = tar_density.log_prob(grid).cpu().numpy().reshape(100, 100)
    plt.figure(figsize=(6, 6))
    plt.contourf(X.cpu(), Y.cpu(), log_probs, levels=50, cmap="rainbow")
    plt.scatter(trajectories[:, -1, 0], trajectories[:, -1, 1], alpha=0.7, color="black", s=10)
    plt.title("Final Distribution of Langevin Samples vs. True Density")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.xlim(-10, 10)
    plt.ylim(-10, 10)
    plt.grid()
    plt.show()


In [20]:
# TODO (Q4.1): Repeat Langevin simulation for another target density (e.g., the symmetric mixture).
#
# After simulating, use plot_final_dist(trajectories, tar_density) to compare samples to the true density.

raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

## Section 5: Langevin dynamics for a multi-modal distribution

### 🧩 Question 5.1 — Mixing across modes
Simulate Langevin dynamics for a **mixture of two Gaussians** where the modes get farther apart.

Try increasing the distance between modes (e.g., 4, 8, 10) and observe:
- Do samples move between modes?
- How does the distance affect mixing?
- What might help (step size, total time, tempering, etc.)?


In [ ]:
# TODO (Q5.1): Define a 2-mode mixture of Gaussians and simulate Langevin dynamics.
#
# Suggested steps:
# 1) build a MixtureOfGaussians with two components whose means are separated by `distance_two_modes`
# 2) run Langevin + EulerMaruyama
# 3) visualize final samples with plot_final_dist(...)
#
# Once it works, try larger distances (see next cells).

raise NotImplementedError("Fill me in!")


### 🧩 Question 5.2 — Experiment

Try with `distance_two_modes = 8.0` and `10.0`.

**What happens? Why?**
Write a short explanation in a markdown cell below (add your own cell).

**Try it with higher dimension**


In [21]:
# TODO (Q5.2): Try distance_two_modes = 8.0
raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!

In [22]:
# TODO (Q5.2): Try distance_two_modes = 10.0
raise NotImplementedError("Fill me in!")


NotImplementedError: Fill me in!